# Broken Links vs. Collection `relatedUrls` Cross-Check

**Goal:** For every collection-level lineage evidence file produced by the lineage skill, take the
`broken_links` field and check whether each broken link is one of the collection's own UMM-C
`relatedUrls` (i.e., a broken link that lives in CMR metadata and needs fixing by the DAAC) or a link
discovered elsewhere during lineage research (ATBDs, landing pages, documents, etc.).

## Steps
1. Walk `Reports/<DataCenter>/<CollectionGroup>/` for the data centers ASF, GES_DISC, GHRC_DAAC,
   LANCEMODIS, LARC_CLOUD, LPCLOUD, LPDAAC_ECS.
   * The per-collection files are named `*_lineage_evidence.json`.
   * `*_suite_lineage_evidence.json` files are **excluded**.
   * The request listed `LPADAAC_ECS`; the actual directory is `LPDAAC_ECS`.
2. Read the top-level `broken_links` field (a `{url: http_status}` dict) and the `target` block
   (short name, version, concept ID).
3. Look up the collection's `relatedUrls` in `eosdis_collections_w_metadata_associations.json`
   (the GraphQL export of 750 EOSDIS collections), matched on `conceptId`.
   * Note: the export stores the link under either `url` or `uRL`, so both keys are read.
4. Compare each broken link to the `relatedUrls`:
   * **exact** match on the URL string, or
   * **normalized** match: case-insensitive, ignoring scheme (`http`/`https`), leading `www.`,
     trailing `/`, and `#fragment`.
5. Write the results to `Reports/Broken_Links_RelatedUrls_CrossCheck.md` in the format
   `Data center` → `* Collection short name` → numbered broken links.

In [1]:
import json, glob, os, re
from collections import defaultdict, Counter

REPO = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'Notebooks' else os.getcwd()
REPORTS = os.path.join(REPO, 'Reports')
COLLECTIONS_FILE = os.path.join(REPO, 'eosdis_collections_w_metadata_associations.json')
OUTPUT_MD = os.path.join(REPORTS, 'Broken_Links_RelatedUrls_CrossCheck.md')

DATA_CENTERS = ['ASF', 'GES_DISC', 'GHRC_DAAC', 'LANCEMODIS', 'LARC_CLOUD', 'LPCLOUD', 'LPDAAC_ECS']
for dc in DATA_CENTERS:
    print(f"{dc:12s} exists={os.path.isdir(os.path.join(REPORTS, dc))}")

ASF          exists=True
GES_DISC     exists=True
GHRC_DAAC    exists=True
LANCEMODIS   exists=True
LARC_CLOUD   exists=True
LPCLOUD      exists=True
LPDAAC_ECS   exists=True

## Step 1 – Locate the collection lineage evidence files

In [2]:
lineage_files = []
for dc in DATA_CENTERS:
    for f in sorted(glob.glob(os.path.join(REPORTS, dc, '*', '*_lineage_evidence.json'))):
        if os.path.basename(f).endswith('_suite_lineage_evidence.json'):
            continue
        lineage_files.append((dc, f))

suite_files = [f for dc in DATA_CENTERS
               for f in glob.glob(os.path.join(REPORTS, dc, '*', '*_suite_lineage_evidence.json'))]
print(f"Collection lineage files: {len(lineage_files)}")
print(f"Suite lineage files excluded: {len(suite_files)}")
print(Counter(dc for dc, _ in lineage_files))

Collection lineage files: 747
Suite lineage files excluded: 23
Counter({'LARC_CLOUD': 523, 'GES_DISC': 157, 'ASF': 41, 'LANCEMODIS': 20, 'LPCLOUD': 3, 'LPDAAC_ECS': 2, 'GHRC_DAAC': 1})

## Step 2 – Load the `relatedUrls` for each collection

In [3]:
with open(COLLECTIONS_FILE) as fh:
    coll_export = json.load(fh)
collections_by_cid = {c['conceptId']: c for c in coll_export['collections']}
print(f"Collections in export: {len(collections_by_cid)} (retrieved {coll_export['retrieved_at_utc']})")
# Note: relatedUrls entries store the link under either `url` or `uRL` (inconsistent key casing)
print(Counter(k for c in collections_by_cid.values() for r in (c.get('relatedUrls') or [])
              for k in r if k.lower() == 'url'))

Collections in export: 750 (retrieved 2026-11-17T16:03:08.655405+00:00)
Counter({'uRL': 11585, 'url': 350})

## Step 3 – Extract `broken_links` and cross-check against `relatedUrls`

In [4]:
def normalize(url):
    u = url.strip().lower()
    u = re.sub(r'^[a-z]+://', '', u)
    u = re.sub(r'^www\.', '', u)
    u = u.split('#', 1)[0]
    return u.rstrip('/')

results = defaultdict(list)   # data center -> list of collection records
missing_in_export = []
files_with_broken = 0
field_presence = Counter()    # (data center, absent/empty/non-empty)

for dc, path in lineage_files:
    with open(path) as fh:
        ev = json.load(fh)
    broken = ev.get('broken_links') or {}
    field_presence[(dc, 'absent' if 'broken_links' not in ev else ('non-empty' if broken else 'empty'))] += 1
    if not broken:
        continue
    files_with_broken += 1
    tgt = ev.get('target', {})
    cid = tgt.get('concept_id')
    coll = collections_by_cid.get(cid)
    if coll is None:
        missing_in_export.append((dc, tgt.get('short_name'), cid))
    # The export uses both `url` and `uRL` as the key for the link value
    related = [dict(r, url=r.get('url') or r.get('uRL')) for r in (coll or {}).get('relatedUrls') or []]
    related = [r for r in related if r['url']]
    exact = {r['url']: r for r in related}
    norm = {normalize(r['url']): r for r in related}

    links = []
    for url, status in broken.items():
        if url in exact:
            match, ru = 'exact', exact[url]
        elif normalize(url) in norm:
            match, ru = 'normalized', norm[normalize(url)]
        else:
            match, ru = None, None
        links.append({
            'url': url, 'status': status, 'match': match,
            'related_url': ru['url'] if ru else None,
            'related_type': ' / '.join(x for x in (ru.get('type'), ru.get('subtype')) if x) if ru else None,
        })
    results[dc].append({
        'short_name': tgt.get('short_name'), 'version': tgt.get('version'),
        'concept_id': cid, 'group': os.path.basename(os.path.dirname(path)),
        'n_related': len(related), 'in_export': coll is not None, 'links': links,
    })

print(f"Lineage files with non-empty broken_links: {files_with_broken} of {len(lineage_files)}")
print(f"Collections not found in the export: {missing_in_export or 'none'}")
print()
print('broken_links field presence per data center:')
for (dc, state), n in sorted(field_presence.items()):
    print(f"  {dc:12s} {state:10s} {n}")

Lineage files with non-empty broken_links: 268 of 747
Collections not found in the export: none

broken_links field presence per data center:
  ASF          absent     41
  GES_DISC     absent     157
  GHRC_DAAC    absent     1
  LANCEMODIS   absent     20
  LARC_CLOUD   absent     248
  LARC_CLOUD   empty      7
  LARC_CLOUD   non-empty  268
  LPCLOUD      absent     3
  LPDAAC_ECS   absent     2

## Step 4 – Summary statistics

In [5]:
rows = []
for dc in DATA_CENTERS:
    recs = results.get(dc, [])
    links = [l for r in recs for l in r['links']]
    rows.append((dc, len(recs), len(links),
                 sum(1 for l in links if l['match']),
                 sum(1 for l in links if not l['match']),
                 sum(1 for r in recs if any(l['match'] for l in r['links']))))

hdr = ('Data center', 'Collections w/ broken links', 'Broken links', 'In relatedUrls',
       'Not in relatedUrls', 'Collections w/ broken relatedUrl')
print(' | '.join(hdr))
for r in rows:
    print(' | '.join(str(x) for x in r))
tot = [sum(r[i] for r in rows) for i in range(1, 6)]
print('TOTAL | ' + ' | '.join(map(str, tot)))

print()
print('Match type:', Counter(l['match'] or 'not listed' for recs in results.values() for r in recs for l in r['links']))
print('HTTP status of broken links:', Counter(str(l['status']) for recs in results.values() for r in recs for l in r['links']))

Data center | Collections w/ broken links | Broken links | In relatedUrls | Not in relatedUrls | Collections w/ broken relatedUrl
ASF | 0 | 0 | 0 | 0 | 0
GES_DISC | 0 | 0 | 0 | 0 | 0
GHRC_DAAC | 0 | 0 | 0 | 0 | 0
LANCEMODIS | 0 | 0 | 0 | 0 | 0
LARC_CLOUD | 268 | 579 | 579 | 0 | 268
LPCLOUD | 0 | 0 | 0 | 0 | 0
LPDAAC_ECS | 0 | 0 | 0 | 0 | 0
TOTAL | 268 | 579 | 579 | 0 | 268

Match type: Counter({'exact': 579})
HTTP status of broken links: Counter({'404': 538, '403': 24, 'ConnectTimeout': 7, 'SSLError': 7, 'ConnectionError': 3})

### Most frequent broken links that are listed in `relatedUrls`

In [6]:
freq = Counter((l['url'], l['related_type']) for recs in results.values() for r in recs
               for l in r['links'] if l['match'])
for (url, typ), n in freq.most_common(25):
    print(f"{n:4d}  {url}  [{typ}]")

 241  https://asdc.larc.nasa.gov/citing-data  [VIEW RELATED INFORMATION / DATA CITATION GUIDELINES]
  34  https://asdc.larc.nasa.gov/project/PREFIRE  [DATA SET LANDING PAGE]
  33  https://asdc.larc.nasa.gov/project/CALIPSO  [PROJECT HOME PAGE]
  24  https://www.jpl.nasa.gov/spaceimages/details.php?id=PIA21927  [VIEW RELATED INFORMATION / MICRO ARTICLE]
  24  https://asdc.larc.nasa.gov/project/MISR  [HOME PAGE]
  17  https://asdc.larc.nasa.gov/project/DSCOVR  [PROJECT HOME PAGE]
  16  https://asdc.larc.nasa.gov/project/CATS-ISS  [VIEW RELATED INFORMATION / GENERAL DOCUMENTATION]
  14  https://www-calipso.larc.nasa.gov/resources/calipso_users_guide/  [VIEW RELATED INFORMATION / USER'S GUIDE]
  12  https://asdc.larc.nasa.gov/project/TEMPO  [VIEW RELATED INFORMATION / GENERAL DOCUMENTATION]
  11  https://www-calipso.larc.nasa.gov/tools/data_avail/  [VIEW RELATED INFORMATION / PROCESSING HISTORY]
  11  https://www-calipso.larc.nasa.gov/resources/calipso_users_guide/faq.php  [VIEW RELATED IN

## Step 5 – Write the results file

In [7]:
def fmt_link(l):
    if l['match']:
        tag = f"**IN relatedUrls** ({l['related_type']}"
        if l['match'] == 'normalized':
            tag += f"; listed as {l['related_url']}"
        tag += ')'
    else:
        tag = 'not in relatedUrls'
    return f"{l['url']} — HTTP {l['status']} — {tag}"

out = ['# Broken Links vs. Collection relatedUrls Cross-Check', '',
       f"Source: `*_lineage_evidence.json` `broken_links` (suite files excluded), compared with "
       f"`relatedUrls` from `eosdis_collections_w_metadata_associations.json`.", '',
       '| ' + ' | '.join(hdr) + ' |', '|' + '---|' * len(hdr)]
out += ['| ' + ' | '.join(map(str, r)) + ' |' for r in rows]
out += ['| **TOTAL** | ' + ' | '.join(map(str, tot)) + ' |', '']

for dc in DATA_CENTERS:
    out.append(f'## {dc}')
    out.append('')
    recs = sorted(results.get(dc, []), key=lambda r: (r['short_name'] or '', r['version'] or ''))
    if not recs:
        if field_presence.get((dc, 'absent')) and not field_presence.get((dc, 'empty')):
            out += ['_No `broken_links` field in any lineage file for this data center (link check not recorded)._', '']
        else:
            out += ['_No broken links found._', '']
        continue
    for r in recs:
        note = '' if r['in_export'] else ' — collection not in export, relatedUrls unavailable'
        out.append(f"* **{r['short_name']}** (v{r['version']}, {r['concept_id']}, group {r['group']}){note}")
        out.append('')
        for i, l in enumerate(r['links'], 1):
            out.append(f"    {i}. {fmt_link(l)}")
        out.append('')

with open(OUTPUT_MD, 'w') as fh:
    fh.write('\n'.join(out))
print(f"Wrote {OUTPUT_MD} ({len(out)} lines)")

Wrote /Users/eleu/dev/esdis/test/metadata-quality-skills/Reports/Broken_Links_RelatedUrls_CrossCheck.md (1424 lines)

### Preview (first data center sections)

In [8]:
print('\n'.join(out[:80]))

# Broken Links vs. Collection relatedUrls Cross-Check

Source: `*_lineage_evidence.json` `broken_links` (suite files excluded), compared with `relatedUrls` from `eosdis_collections_w_metadata_associations.json`.

| Data center | Collections w/ broken links | Broken links | In relatedUrls | Not in relatedUrls | Collections w/ broken relatedUrl |
|---|---|---|---|---|---|
| ASF | 0 | 0 | 0 | 0 | 0 |
| GES_DISC | 0 | 0 | 0 | 0 | 0 |
| GHRC_DAAC | 0 | 0 | 0 | 0 | 0 |
| LANCEMODIS | 0 | 0 | 0 | 0 | 0 |
| LARC_CLOUD | 268 | 579 | 579 | 0 | 268 |
| LPCLOUD | 0 | 0 | 0 | 0 | 0 |
| LPDAAC_ECS | 0 | 0 | 0 | 0 | 0 |
| **TOTAL** | 268 | 579 | 579 | 0 | 268 |

## ASF

_No `broken_links` field in any lineage file for this data center (link check not recorded)._

## GES_DISC

_No `broken_links` field in any lineage file for this data center (link check not recorded)._

## GHRC_DAAC

_No `broken_links` field in any lineage file for this data center (link check not recorded)._

## LANCEMODIS

_No `brok

## Summary

**Scope:** 747 collection-level `*_lineage_evidence.json` files across 7 data centers (23 suite files excluded).
All 747 collections were found in `eosdis_collections_w_metadata_associations.json`.

**Key findings**
* **Only LARC_CLOUD has link-check results.** The `broken_links` field appears in 275 of the 523 LARC_CLOUD
  files (268 non-empty, 7 empty). It is **absent** from every ASF, GES_DISC, GHRC_DAAC, LANCEMODIS, LPCLOUD, and
  LPDAAC_ECS file, and from 248 LARC_CLOUD files. Those collections weren't link-checked, so "no entry" there
  does not mean "no broken links".
* **268 LARC_CLOUD collections have 579 broken links in total.**
* **All 579 broken links (100%) are exact matches to a URL in the collection's own UMM-C `relatedUrls`.**
  The normalized match never had to be used. So every reported broken link is a CMR metadata problem
  the DAAC can fix.
* HTTP outcomes: 538 × 404, 24 × 403, 7 × ConnectTimeout, 7 × SSLError, 3 × ConnectionError.
  Check the 403 and connection errors by hand; they may be bot blocking or temporary outages.
* The breakage is concentrated in a few shared URLs. The biggest is `https://asdc.larc.nasa.gov/citing-data`
  (DATA CITATION GUIDELINES), broken in 248 collections. Next are the ASDC `/project/<MISSION>` pages
  (PREFIRE, CALIPSO, MISR, DSCOVR, CATS-ISS, TEMPO, …) and the legacy `www-calipso.larc.nasa.gov` user's guide pages.
  Fixing these few URLs at the source (or with redirects) would clear most of the findings.

**Data quirk:** the GraphQL export stores the related-URL value under either `url` (350 entries) or `uRL` (11,585
entries). Reading only `url` would wrongly report almost every link as "not in relatedUrls".

**Output:** `Reports/Broken_Links_RelatedUrls_CrossCheck.md`